<a href="https://colab.research.google.com/github/andluizsouza/unicamp-llm-agents/blob/main/modules/01_fundamentos_ia_pln/C_uso_ajuste_llm/hands_on_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploração do dataset SmolTalk2 e análise do comprimento de tokens

Este notebook cobre três etapas principais:
1. **Instalação das dependências** necessárias para trabalhar com modelos e datasets do Hugging Face.
2. **Exploração do dataset** `HuggingFaceTB/smoltalk2` sem baixá-lo completamente: seus splits, tamanhos e estrutura.
3. **Análise do comprimento de tokens** sobre uma amostra do split selecionado, usando o tokenizer do Gemma-3 para estimar o valor ideal de `max_length` antes de fazer fine-tuning.

> **Ambiente:** Google Colab com GPU (T4 ou superior). O notebook requer acesso ao Hugging Face para baixar o dataset e o tokenizer.

## 🔑 Pré-requisito: Token do Hugging Face

Alguns modelos e datasets no Hugging Face requerem autenticação. Neste notebook é usado o modelo `google/gemma-3-1b-it`, que exige aceitar os termos de uso e ter um token válido.

### Como obter o seu token?

- **Passo 1.** Crie uma conta (gratuita) em [https://huggingface.co/join](https://huggingface.co/join).

- **Passo 2.** Vá ao seu perfil → **Settings** → **Access Tokens**: [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

- **Passo 3.** Clique em **"New token"**, atribua um nome (por exemplo `colab-token`), selecione o papel **"Read"** e clique em **"Generate a token"**. Copie o token gerado (começa com `hf_...`).

- **Passo 4.** Aceite os termos de uso do modelo Gemma: acesse [https://huggingface.co/google/gemma-3-1b-it](https://huggingface.co/google/gemma-3-1b-it) e clique em **"Agree and access repository"** (é necessário estar logado).

- **Passo 5.** Na célula de código correspondente (mais adiante no notebook), substitua o valor de `HF_TOKEN` pelo seu token pessoal:
```python
HF_TOKEN = "hf_SEU_TOKEN_AQUI"
```

## 1. Instalação das dependências

Instalamos as bibliotecas necessárias:
- **`bitsandbytes`**: quantização de modelos (reduz o uso de memória da GPU).
- **`peft`**: Parameter-Efficient Fine-Tuning (LoRA, etc.).
- **`trl`**: treinamento de LLMs com reforço; inclui o `SFTTrainer`.
- **`accelerate`**: abstração multi-GPU / mixed precision.
- **`datasets`**: carregamento e manipulação de datasets do Hugging Face.
- **`transformers`**: modelos, tokenizers e pipelines do Hugging Face.

O flag `-q` suprime a saída verbosa e `-U` atualiza para a versão mais recente, caso já estejam instaladas.

In [ ]:
!pip3 install -q -U bitsandbytes peft trl accelerate datasets transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 721.6/721.6 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.0 MB/s eta 0:00:00


## 2. Exploração do dataset `smoltalk2`

O dataset [`HuggingFaceTB/smoltalk2`](https://huggingface.co/datasets/HuggingFaceTB/smoltalk2) é uma coleção multilíngue de conversas no formato instrução-resposta, projetada para fine-tuning supervisionado (SFT) de LLMs.

Antes de baixar os dados, convém inspecionar a estrutura do dataset: quais splits existem, quantos exemplos cada um tem e quanto espaço ocupa. As funções da biblioteca `datasets` permitem fazer isso **sem baixar os dados completos**.

### 2.1 Listar os splits disponíveis

Um *split* é uma partição do dataset (por exemplo `train`, `test`, `validation`). O `smoltalk2` tem uma configuração chamada `"SFT"` que agrupa os splits de treinamento supervisionado. A função `get_dataset_split_names` retorna os nomes de todos os splits disponíveis sem baixar nenhum dado.

In [ ]:
from datasets import get_dataset_split_names

get_dataset_split_names("HuggingFaceTB/smoltalk2", "SFT")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

['LongAlign_64k_Qwen3_32B_yarn_131k_think',
 'OpenThoughts3_1.2M_think',
 'aya_dataset_Qwen3_32B_think',
 'multi_turn_reasoning_if_think',
 's1k_1.1_think',
 'smolagents_toolcalling_traces_think',
 'smoltalk_multilingual8_Qwen3_32B_think',
 'smoltalk_systemchats_Qwen3_32B_think',
 'table_gpt_Qwen3_32B_think',
 'LongAlign_64k_context_lang_annotated_lang_6_no_think',
 'Mixture_of_Thoughts_science_no_think',
 'OpenHermes_2.5_no_think',
 'OpenThoughts3_1.2M_no_think_no_think',
 'hermes_function_calling_v1_no_think',
 'smoltalk_multilingual_8languages_lang_5_no_think',
 'smoltalk_smollm3_everyday_conversations_no_think',
 'smoltalk_smollm3_explore_instruct_rewriting_no_think',
 'smoltalk_smollm3_smol_magpie_ultra_no_think',
 'smoltalk_smollm3_smol_rewrite_no_think',
 'smoltalk_smollm3_smol_summarize_no_think',
 'smoltalk_smollm3_systemchats_30k_no_think',
 'table_gpt_no_think',
 'tulu_3_sft_personas_instruction_following_no_think',
 'xlam_traces_no_think',
 'smoltalk_everyday_convs_reasonin

### 2.2 Inspecionar os metadados do dataset

`load_dataset_builder` baixa apenas o arquivo de configuração e os metadados do dataset (alguns KB), sem os dados reais. Isso nos permite acessar informações como o esquema de colunas, descrição, splits e tamanhos.

In [ ]:
# load_dataset_builder baixa apenas a configuração do dataset (metadados)
# sem baixar os exemplos reais — útil para exploração eficiente
# Ver info del dataset sin descargar
from datasets import load_dataset_builder

builder = load_dataset_builder("HuggingFaceTB/smoltalk2", "SFT")
print(builder.info)

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

DatasetInfo(features={'messages': List({'content': Value('string'), 'role': Value('string')}), 'chat_template_kwargs': {'custom_instructions': Value('string'), 'enable_thinking': Value('bool'), 'python_tools': List(Value('string')), 'xml_tools': List(Value('string'))}, 'source': Value('string')}, builder_name='parquet', dataset_name='smoltalk2', config_name='SFT', version=0.0.0, splits={'LongAlign_64k_Qwen3_32B_yarn_131k_think': SplitInfo(name='LongAlign_64k_Qwen3_32B_yarn_131k_think', num_bytes=520907823, num_examples=7526), 'OpenThoughts3_1.2M_think': SplitInfo(name='OpenThoughts3_1.2M_think', num_bytes=56273098407, num_examples=1133524), 'aya_dataset_Qwen3_32B_think': SplitInfo(name='aya_dataset_Qwen3_32B_think', num_bytes=60172886, num_examples=15222), 'multi_turn_reasoning_if_think': SplitInfo(name='multi_turn_reasoning_if_think', num_bytes=421909719, num_examples=28217), 's1k_1.1_think': SplitInfo(name='s1k_1.1_think', num_bytes=25598191, num_examples=835), 'smolagents_toolcallin

### 2.3 Ver os splits e seus tamanhos

`builder.info.splits` é um dicionário onde cada chave é o nome de um split e o valor contém metadados como `num_examples` (quantidade de exemplos) e `num_bytes` (tamanho em bytes).

In [ ]:
# Ver todos los splits disponibles y su tamaño
print(builder.info.splits)

{'LongAlign_64k_Qwen3_32B_yarn_131k_think': SplitInfo(name='LongAlign_64k_Qwen3_32B_yarn_131k_think', num_bytes=520907823, num_examples=7526), 'OpenThoughts3_1.2M_think': SplitInfo(name='OpenThoughts3_1.2M_think', num_bytes=56273098407, num_examples=1133524), 'aya_dataset_Qwen3_32B_think': SplitInfo(name='aya_dataset_Qwen3_32B_think', num_bytes=60172886, num_examples=15222), 'multi_turn_reasoning_if_think': SplitInfo(name='multi_turn_reasoning_if_think', num_bytes=421909719, num_examples=28217), 's1k_1.1_think': SplitInfo(name='s1k_1.1_think', num_bytes=25598191, num_examples=835), 'smolagents_toolcalling_traces_think': SplitInfo(name='smolagents_toolcalling_traces_think', num_bytes=200401637, num_examples=9079), 'smoltalk_multilingual8_Qwen3_32B_think': SplitInfo(name='smoltalk_multilingual8_Qwen3_32B_think', num_bytes=1900647658, num_examples=244736), 'smoltalk_systemchats_Qwen3_32B_think': SplitInfo(name='smoltalk_systemchats_Qwen3_32B_think', num_bytes=123542086, num_examples=27436

### 2.4 Tabela resumo dos splits

Construímos um `DataFrame` do pandas para visualizar de forma organizada quantos exemplos e quantos MB ocupa cada split. Isso é útil para decidir com qual partição trabalhar e qual tamanho de amostra é razoável para um experimento.

In [ ]:
# Construímos uma tabela com nome do split, quantidade de exemplos e tamanho em MB
# Ordenamos de maior para menor por número de exemplos para identificar os splits maiores
import pandas as pd

splits_info = builder.info.splits

df = pd.DataFrame([
    {
        "split": name,
        "num_examples": info.num_examples,
        "size_MB": round(info.num_bytes / 1024**2, 1)
    }
    for name, info in splits_info.items()
]).sort_values("num_examples", ascending=False)

df

,split,num_examples,size_MB
1,OpenThoughts3_1.2M_think,1133524,53666.2
12,OpenThoughts3_1.2M_no_think_no_think,435193,1158.8
17,smoltalk_smollm3_smol_magpie_ultra_no_think,406843,2690.2
11,OpenHermes_2.5_no_think,384900,558.3
14,smoltalk_multilingual_8languages_lang_5_no_think,254047,538.8
6,smoltalk_multilingual8_Qwen3_32B_think,244736,1812.6
19,smoltalk_smollm3_smol_summarize_no_think,96061,218.5
10,Mixture_of_Thoughts_science_no_think,86110,124.2
23,xlam_traces_no_think,59962,92.3
18,smoltalk_smollm3_smol_rewrite_no_think,53262,85.4


## 3. Seleção do split de trabalho

A partir da tabela anterior, selecionamos o split `smoltalk_multilingual_8languages_lang_5_no_think`. Este split contém conversas multilíngues em 8 idiomas, sem ativação do modo de raciocínio (*thinking*), o que o torna adequado para fine-tuning de instrução padrão.

O split tem **254.047 exemplos**. Para um experimento exploratório, trabalhar com 1% dos dados (~2.540 exemplos) é suficiente para medir a distribuição de comprimentos sem o custo de baixar o dataset completo.

> **Split selecionado:** `smoltalk_multilingual_8languages_lang_5_no_think`

### 3.1 Calcular o tamanho da amostra (1% do split)

Calculamos 1% do total de exemplos do split para saber quantos exemplos baixar. O resultado (~2.540) será o número de linhas que carregaremos usando streaming.

In [ ]:
# 1% de 254.047 exemplos = ~2.540 exemplos a baixar como amostra
1 * (254047 / 100)

2540.47

## 4. Carregamento de uma amostra do dataset

Em vez de baixar as 254k linhas completas, usamos **streaming** para ler o dataset de forma progressiva (linha a linha) e pegar apenas os primeiros 2.540 exemplos.

### Por que streaming?

Com `streaming=True`, os dados **não são baixados todos de uma vez**. O dataset é lido como um fluxo (*IterableDataset*), o que permite pegar uma amostra com `islice` sem esperar o download completo do dataset.

Em seguida, convertemos essa amostra em um `Dataset` estático com `Dataset.from_list()`, que fica em memória e permite indexação direta (necessária para as células seguintes).

In [ ]:
# Carregamos o split em modo streaming: os dados são lidos progressivamente, não todos de uma vez
# Isso evita baixar o dataset completo (vários GB) quando precisamos apenas de uma amostra
from datasets import load_dataset
from itertools import islice

ds_stream = load_dataset(
    "HuggingFaceTB/smoltalk2",
    "SFT",
    split="smoltalk_multilingual_8languages_lang_5_no_think",
    streaming=True  # <-- modo streaming: não baixa todo o dataset
)

# islice(ds_stream, 2540) pega os primeiros 2540 exemplos do fluxo
# Dataset.from_list() converte a lista em um Dataset indexável em memória
from datasets import Dataset
sample_ds = Dataset.from_list(list(islice(ds_stream, 2540)))

Resolving data files:   0%|          | 0/124 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/113 [00:00<?, ?it/s]

## 5. Inspeção da amostra

Verificamos que o dataset foi carregado corretamente e exploramos sua estrutura. Cada exemplo (`sample_ds[i]`) é um dicionário com uma chave `messages`, que contém uma lista de turnos de conversa no formato `[{role, content}, ...]`. Esse é o formato padrão de chat para fine-tuning com `SFTTrainer`.

In [ ]:
# Exibe o resumo do Dataset: número de linhas, colunas e tipos de dados
sample_ds

Dataset({
    features: ['messages', 'chat_template_kwargs', 'source'],
    num_rows: 2540
})

In [ ]:
# Inspecionamos o primeiro exemplo do dataset
# Cada exemplo tem uma chave 'messages' com a conversa no formato de chat
sample_ds[0]

{'messages': [{'content': 'O diretor da escola, juntamente com os professores e demais funcionários, ficou muito satisfeito ao concluir com sucesso a difícil tarefa de organizar o festival escolar anual dentro do prazo, que foi muito apreciado pelos estudantes e seus responsáveis, que também estavam presentes.',
   'role': 'user'},
  {'content': 'O diretor da escola, em conjunto com os professores e demais funcionários, sentiu-se extremamente satisfeito ao concluir com êxito a árdua tarefa de organizar o festival escolar anual dentro do prazo estabelecido. O evento, que foi muito apreciado pelos estudantes e seus responsáveis, que também participaram ativamente, demonstrou o empenho e a dedicação de toda a equipe envolvida. A realização do festival não apenas proporcionou momentos de lazer e entretenimento, mas também fortaleceu o vínculo entre a comunidade escolar e as famílias, reafirmando o compromisso da instituição em promover atividades que enriqueçam a experiência educacional do

In [ ]:
# Inspecionamos o segundo exemplo para verificar que o formato é consistente
sample_ds[1]

{'messages': [{'content': "J'aimerais commencer le vélotourisme.",
   'role': 'user'},
  {'content': "Le vélotourisme est une excellente façon de découvrir de nouveaux paysages, de vivre des aventures et de se maintenir en forme. Pour bien commencer, il est important de bien se préparer. Commencez par choisir un vélo adapté à vos besoins et à vos aspirations. Un vélo de route sera plus adapté pour les longues distances sur des routes bien entretenues, tandis qu'un vélo de voyage ou un VTT sera plus polyvalent et vous permettra de vous aventurer sur des chemins plus accidentés.\n\nIl est également essentiel de bien s'équiper. Investissez dans un bon sac à dos ou des sacoches pour transporter vos affaires, et n'oubliez pas les accessoires de sécurité comme un casque, des gants, des lunettes de soleil et des vêtements de pluie. Un kit de réparation de base, comprenant des chambres à air de rechange, des outils et une pompe, est indispensable. En outre, une carte détaillée de votre itinéra

## 6. Carregamento do tokenizer

O tokenizer converte o texto das conversas em sequências de *tokens* (números inteiros) que o modelo pode processar.

Usamos o tokenizer da versão **IT** (*Instruction-Tuned*) do `google/gemma-3-1b`, que inclui o `chat_template` configurado corretamente para conversas multi-turno. Este é o mesmo tokenizer que o `SFTTrainer` usará internamente durante o treinamento.

> ⚠️ **Lembre-se de substituir `HF_TOKEN`** pelo seu token pessoal do Hugging Face (consulte a seção de pré-requisitos no início do notebook).

In [ ]:
from transformers import AutoTokenizer

# ⚠️ Substitua este valor pelo seu token pessoal do Hugging Face
# Consulte a seção 'Pré-requisito' no início do notebook para obtê-lo
HF_TOKEN = "hf_SEU_TOKEN_AQUI"  # <-- substitua pelo seu token

# Carregamos o tokenizer do modelo IT (Instruction-Tuned)
# O tokenizer IT tem o chat_template configurado corretamente para o Gemma
# Usamos o IT porque ele tem o chat template de instruções configurado
tokenizer = AutoTokenizer.from_pretrained(
    "google/gemma-3-1b-it",   # IT = Instruction-Tuned
    token=HF_TOKEN
)

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

## 7. Análise da distribuição de comprimentos em tokens

Antes de fazer fine-tuning, é importante conhecer a distribuição de comprimentos (em tokens) dos exemplos do dataset. Isso nos permite escolher um valor adequado para `max_length` no trainer:

- Se `max_length` for **muito pequeno**: muitos exemplos serão truncados e informação será perdida.
- Se `max_length` for **muito grande**: memória da GPU é desperdiçada com padding desnecessário.

### Processo de medição

Para cada exemplo:
1. Aplica-se o `chat_template` para converter a lista de mensagens no formato de texto que o Gemma espera como entrada (incluindo tokens especiais de início/fim de turno).
2. Tokeniza-se o texto resultante para contar o número de tokens.

Ao final, calculam-se estatísticas descritivas (mínimo, mediana, média, máximo) sobre a distribuição de comprimentos.

In [ ]:
import numpy as np
from tqdm import tqdm

def count_tokens_in_split(dataset_split, tokenizer, sample_size=None):
    """
    Mide la distribución de longitud (en tokens) de los ejemplos del dataset
    después de aplicar el chat template.

    Args:
        dataset_split: partición del dataset a medir
        tokenizer: tokenizer del modelo
        sample_size: si se especifica, mide solo una muestra aleatoria

    Returns:
        np.array con la cantidad de tokens de cada ejemplo
    """
    if sample_size and sample_size < len(dataset_split):
        # Si se pide una muestra, la barajamos aleatoriamente antes de seleccionar
        dataset_split = dataset_split.shuffle(seed=42).select(range(sample_size))

    lengths = []
    for example in tqdm(dataset_split, desc="Contando tokens"):
        # Aplicar el mismo template que usará SFTTrainer internamente
        # add_generation_prompt=False porque la respuesta del assistant YA está incluida
        text = tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
        )
        # Tokenizar el texto para contar sus tokens
        # add_special_tokens=False: el template ya incluye los tokens especiales
        tokens = tokenizer(text, add_special_tokens=False)["input_ids"]
        lengths.append(len(tokens))

    return np.array(lengths)

# Medir la distribución de tokens en todo el split de train
lengths = count_tokens_in_split(
    sample_ds,
    tokenizer,
)

# Mostrar estadísticas de la distribución
print(f"\n📊 Distribución de longitudes (en tokens):")
print(f"   Ejemplos medidos: {len(lengths)}")
print(f"   Mínimo:           {lengths.min()} tokens")
print(f"   Mediana:          {int(np.median(lengths))} tokens")
print(f"   Media:            {int(lengths.mean())} tokens")
print(f"   Máximo:           {lengths.max()} tokens")
print(f"\n→ Usaremos max_length=507 en el trainer, que cubre la mayoría de ejemplos.")

Contando tokens: 100%|██████████| 2540/2540 [00:03<00:00, 644.85it/s]



📊 Distribución de longitudes (en tokens):
   Ejemplos medidos: 2540
   Mínimo:           30 tokens
   Mediana:          495 tokens
   Media:            507 tokens
   Máximo:           1918 tokens

→ Usaremos max_length=507 en el trainer, que cubre la mayoría de ejemplos.


## ✅ Resumo

Com este notebook:

- Exploramos o dataset `smoltalk2` sem baixá-lo completamente.
- Selecionamos e baixamos via streaming uma amostra de 1% do split multilíngue.
- Carregamos o tokenizer do Gemma-3-1b-it e aplicamos o chat template.
- Medimos a distribuição de comprimentos em tokens para calibrar o parâmetro `max_length`.

O valor de `max_length` obtido (507 tokens para este split) pode ser usado diretamente ao configurar o `SFTTrainer` na etapa de fine-tuning.